# ML-Driven PVT Variation Aware Leakage Estimation (Fine-Tuned)
## Case: PMOS Transistor — ON State (Conduction) (PMOS ON)

### Project Overview
In advanced semiconductor technology nodes (such as 22nm High-K/Metal-Gate), Process, Voltage, and Temperature (PVT) variations severely impact transistor leakage and conduction currents.
This notebook develops a **fine-tuned LightGBM (Light Gradient Boosting Machine)** model with **Physics-Informed Feature Engineering** to estimate all four terminal currents ($I_d, I_g, I_s, I_b$) from single-transistor operating conditions and process parameters, aiming for **up to 99% accuracy**.

### Transistor Operating Case
- **Device Type**: `PMOS`
- **Operating Regime**: `ON`
- **Description**: Gate is biased to 0V with body at VDD (Vg = 0, Vb = VDD). Channel is strongly inverted; simulates conduction and gate tunneling under PVT variations.

---
### Key Accuracy Enhancements Implemented:
1. **Physics-Informed Features**: Added $V_{ds} = V_d - V_s$, $|V_{ds}|$, $V_{gs}$, $V_{gd}$, $V_{bs}$, $V_{ov} = V_{gs} - V_{th0}$, thermal voltage $V_t$, $W/L$, and oxide electric field $E_{ox}$.
2. **Singularity Resolution**: Eliminated decision-tree staircase errors along the $V_d \approx V_s$ boundary.
3. **Deep Tree Architecture**: Upgraded to `num_leaves=127`, `max_depth=12`, `learning_rate=0.02`, and `n_estimators=2500` with early stopping.
4. **SPICE-Standard Error Formulation**: Applied standard signal-relative error floor to prevent near-zero division explosion.


In [ ]:
# 1. Environment Setup & Library Imports
import os
import sys
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# LightGBM and Scikit-Learn
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Configure plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Set global random seed for exact reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment setup complete. LightGBM version:", lgb.__version__)


In [ ]:
# 2. Configuration & Directory Paths
CASE_NAME = "pmos_on"
RANDOM_STATE = 42

# Detect directory of the notebook / case folder
if "__file__" in locals():
    NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__file__))
else:
    # In Jupyter, detect whether run from case subfolder, ml_model/, or project root
    if os.path.exists("plots") and os.path.exists("saved_models"):
        NOTEBOOK_DIR = os.path.abspath(".")
    elif os.path.exists(os.path.join("ml_model", CASE_NAME.upper())):
        NOTEBOOK_DIR = os.path.abspath(os.path.join("ml_model", CASE_NAME.upper()))
    elif os.path.exists(CASE_NAME.upper()):
        NOTEBOOK_DIR = os.path.abspath(CASE_NAME.upper())
    else:
        NOTEBOOK_DIR = os.path.abspath(".")

SAVED_MODELS_DIR = os.path.join(NOTEBOOK_DIR, "saved_models")
PLOTS_DIR = os.path.join(NOTEBOOK_DIR, "plots")
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# =========================================================================================
# DATASET SELECTION: Choose which dataset to train on
# =========================================================================================
candidate_paths = [
    os.path.join(NOTEBOOK_DIR, "..", "..", "dataset", CASE_NAME.upper(), f"{CASE_NAME}.csv"),
    os.path.join(NOTEBOOK_DIR, "..", "..", "Data_Generation_M1", "data", f"{CASE_NAME}.csv"),
    os.path.join("..", "..", "dataset", CASE_NAME.upper(), f"{CASE_NAME}.csv"),
    os.path.join("..", "dataset", CASE_NAME.upper(), f"{CASE_NAME}.csv"),
    os.path.join("dataset", CASE_NAME.upper(), f"{CASE_NAME}.csv"),
    os.path.join("..", "..", "Data_Generation_M1", "data", f"{CASE_NAME}.csv"),
    os.path.join("..", "Data_Generation_M1", "data", f"{CASE_NAME}.csv"),
    os.path.join("Data_Generation_M1", "data", f"{CASE_NAME}.csv"),
]

DATA_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        DATA_PATH = os.path.abspath(p)
        break

if DATA_PATH is None:
    raise FileNotFoundError(f"Could not locate dataset for '{CASE_NAME}'. Checked paths: {candidate_paths}")

print(f"Dataset selected : {DATA_PATH}")
print(f"Model save path  : {SAVED_MODELS_DIR}")
print(f"Plots save path  : {PLOTS_DIR}")


In [ ]:
# 3. Data Loading & Initial Inspection
df_raw = pd.read_csv(DATA_PATH)

# Drop auxiliary columns like 'itotal' if present from simulation scripts
if 'itotal' in df_raw.columns:
    df_raw = df_raw.drop(columns=['itotal'])

print(f"Loaded {len(df_raw)} samples with {len(df_raw.columns)} columns.")
print("\nDataset preview (first 5 samples):")
display(df_raw.head())

print("\nMissing value check:")
missing_cnt = df_raw.isnull().sum().sum()
print(f"  Total missing values: {missing_cnt}")

# Validate Kirchhoff's Current Law (KCL): Id + Ig + Is + Ib ≈ 0
kcl_residual = np.abs(df_raw['id'] + df_raw['ig'] + df_raw['is'] + df_raw['ib'])
print(f"\nKirchhoff's Current Law (KCL) check on SPICE training data:")
print(f"  Max absolute KCL residual  : {kcl_residual.max():.3e} A")
print(f"  Mean absolute KCL residual : {kcl_residual.mean():.3e} A")


## 4. Physics-Informed Feature Engineering

### Why Derived Physical Features Are Essential:
A MOSFET does not operate on absolute voltages $V_d, V_s, V_g, V_b$ in isolation. Instead, carrier transport, channel inversion, and junction electric fields depend strictly on **potential differences**:
1. **$V_{ds} = V_d - V_s$**: Driving potential for channel drift & diffusion.
2. **$|V_{ds}|$**: Magnitude of channel bias (symmetric conduction).
3. **$V_{gs} = V_g - V_s$**: Gate-to-source bias controlling channel inversion.
4. **$V_{gd} = V_g - V_d$**: Gate-to-drain bias controlling channel pinch-off.
5. **$V_{bs} = V_b - V_s$**: Body-to-source bias controlling substrate depletion / body effect.
6. **$V_{bd} = V_b - V_d$**: Body-to-drain bias controlling reverse PN junction leakage.
7. **$V_{ov} = V_{gs} - V_{	ext{th0}}$**: Effective gate overdrive voltage.
8. **$T_K = T + 273.15$**: Absolute temperature in Kelvin.
9. **$V_t = rac{k \cdot T_K}{q}$**: Thermal voltage ($\sim 25	ext{–}35	ext{ mV}$), governing exponential subthreshold slopes.
10. **$W/L = W / 22	ext{nm}$**: Transistor aspect ratio.
11. **$E_{ox} = rac{V_g - V_b}{t_{	ext{oxe}}}$**: Oxide electric field governing quantum tunneling.

By providing these features directly, decision trees no longer need to approximate continuous diagonal curves with hundreds of staircase splits.


In [ ]:
# 4. Feature Engineering: Adding Physics-Informed Features
df = df_raw.copy()

# 1. Potential differences
df['vds'] = df['vd'] - df['vs']
df['abs_vds'] = np.abs(df['vds'])
df['vgs'] = df['vg'] - df['vs']
df['vgd'] = df['vg'] - df['vd']
df['vbs'] = df['vb'] - df['vs']
df['vbd'] = df['vb'] - df['vd']

# 2. Gate overdrive voltage
df['vov'] = df['vgs'] - df['vth0']

# 3. Temperature in Kelvin & Thermal voltage Vt = k*T/q
df['temp_k'] = df['temp'] + 273.15
K_BOLTZMANN_OVER_Q = 8.617333e-5  # V / K
df['vt'] = K_BOLTZMANN_OVER_Q * df['temp_k']

# 4. Geometric aspect ratio (Lmin = 22nm)
L_MIN = 22e-9
df['wl_ratio'] = df['w'] / L_MIN

# 5. Oxide electric field
df['e_ox'] = (df['vg'] - df['vb']) / df['toxe']

BASE_FEATURES = ['vdd', 'temp', 'w', 'vd', 'vs', 'vg', 'vb', 'vth0', 'toxe', 'ndep', 'u0', 'xj']
PHYSICS_FEATURES = ['vds', 'abs_vds', 'vgs', 'vgd', 'vbs', 'vbd', 'vov', 'temp_k', 'vt', 'wl_ratio', 'e_ox']
FEATURE_COLS = BASE_FEATURES + PHYSICS_FEATURES
TARGET_COLS = ['id', 'ig', 'is', 'ib']

print(f"Total input features : {len(FEATURE_COLS)} ({len(BASE_FEATURES)} base + {len(PHYSICS_FEATURES)} physics-informed)")
print(f"Features: {FEATURE_COLS}")


In [ ]:
# 5. Log-Transformation of Terminal Currents
EPSILON = 1e-20  # Numerical floor below SPICE resolution (1e-18 A)

for col in TARGET_COLS:
    df[f"{col}_log"] = np.log10(np.abs(df[col]) + EPSILON)
    df[f"{col}_sign"] = np.sign(df[col]).replace(0, 1)

print("Summary of log-transformed targets:")
display(df[[f"{c}_log" for c in TARGET_COLS]].describe().T[['mean', 'std', 'min', '50%', 'max']])

# Visualize target distributions
fig, axes = plt.subplots(2, 4, figsize=(18, 6))
for idx, col in enumerate(TARGET_COLS):
    axes[0, idx].hist(df[col], bins=40, color='royalblue', edgecolor='black', alpha=0.7)
    axes[0, idx].set_title(f"Raw {col} (Amperes)")
    axes[0, idx].set_yscale('log')
    axes[0, idx].set_xlabel("Current (A)")
    
    axes[1, idx].hist(df[f"{col}_log"], bins=40, color='darkorange', edgecolor='black', alpha=0.7)
    axes[1, idx].set_title(f"Log10(|{col}| + 1e-20)")
    axes[1, idx].set_xlabel("log10(|I|)")

plt.suptitle(f"{CASE_NAME.upper()} — Current Distributions (Raw vs. Log-Transformed)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 6. Dataset Splitting (70% Train, 15% Validation, 15% Test)
X = df[FEATURE_COLS]
y_logs = df[[f"{c}_log" for c in TARGET_COLS]]
y_signs = df[[f"{c}_sign" for c in TARGET_COLS]]
y_raw = df[TARGET_COLS]

# Split into 70% train and 30% temp
X_train, X_temp, y_log_train, y_log_temp, y_raw_train, y_raw_temp, y_sign_train, y_sign_temp = train_test_split(
    X, y_logs, y_raw, y_signs,
    test_size=0.30,
    random_state=RANDOM_STATE,
    shuffle=True
)

# Split temp into 15% validation and 15% test
X_val, X_test, y_log_val, y_log_test, y_raw_val, y_raw_test, y_sign_val, y_sign_test = train_test_split(
    X_temp, y_log_temp, y_raw_temp, y_sign_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Dataset partitions for {CASE_NAME}:")
print(f"  Training set   : {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.1f}%)")
print(f"  Validation set : {X_val.shape[0]} samples ({X_val.shape[0]/len(df)*100:.1f}%)")
print(f"  Test set       : {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.1f}%)")


## 7. Fine-Tuned LightGBM Model Architecture

### Hyperparameter Fine-Tuning:
- **`num_leaves: 127`**: Expanded leaf count allows learning high-order interactions among PVT parameters.
- **`max_depth: 12`**: Deep enough to resolve BSIM4 saturation and subthreshold boundaries.
- **`learning_rate: 0.02`**: Finer gradient descent steps prevent overshooting.
- **`n_estimators: 2500`**: Extended boosting iterations with early stopping at 80 rounds.
- **`min_child_samples: 15`**: Provides fine granularity around transition regions.
- **`reg_alpha: 1e-3` & `reg_lambda: 1e-2`**: L1/L2 regularization to prevent overfitting.


In [ ]:
# 7. Model Training with Fine-Tuned LightGBM
LGBM_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'num_leaves': 127,
    'max_depth': 12,
    'min_child_samples': 15,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 1e-3,
    'reg_lambda': 1e-2,
    'n_estimators': 2500,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbose': -1
}

trained_models = {}

print(f"Training fine-tuned LightGBM models for {CASE_NAME.upper()}...")
start_total = time.time()

for target in TARGET_COLS:
    target_log_name = f"{target}_log"
    print(f"\n--- Training LightGBM for target: {target.upper()} ({target_log_name}) ---")
    
    model = lgb.LGBMRegressor(**LGBM_PARAMS)
    
    # Train with early stopping monitored on validation set
    model.fit(
        X_train, y_log_train[target_log_name],
        eval_set=[(X_train, y_log_train[target_log_name]), (X_val, y_log_val[target_log_name])],
        eval_names=['train', 'val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=80, verbose=False),
            lgb.log_evaluation(period=400)
        ]
    )
    
    best_iter = model.best_iteration_ if hasattr(model, 'best_iteration_') else model.n_estimators
    print(f"✓ {target.upper()} training complete. Best iteration: {best_iter}")
    trained_models[target] = model

print(f"\nAll 4 models trained successfully in {time.time() - start_total:.2f} seconds.")


## 8. Model Accuracy Evaluation

### Standard SPICE Relative Error Metric:
When evaluating electrical currents spanning many orders of magnitude, standard SPICE practice (`ABSTOL`, `RELTOL`) includes a practical signal floor:
$$\text{Relative Error} = \frac{|I_{\text{pred}} - I_{\text{SPICE}}|}{|I_{\text{SPICE}}| + I_{\text{floor}}} \times 100\%$$
- For conduction currents ($I_d, I_s$): $I_{\text{floor}} = 10^{-10}\text{ A}$ ($0.0001\%$ of full-scale $10^{-4}\text{ A}$). This prevents artificial division-by-zero explosions when $V_d \approx V_s$.
- For leakage currents ($I_g, I_b$): $I_{\text{floor}} = 10^{-15}\text{ A}$ (femtoampere measurement floor).


In [ ]:
# 8. Performance Evaluation on Validation and Test Sets

def evaluate_models(models, X_eval, y_log_eval, y_raw_eval, y_sign_eval, split_name="Validation"):
    """
    Computes all standard and project-specific accuracy metrics:
      - Mean Accuracy (%) = 100% - Mean Relative Error (%)
      - Median Accuracy (%) = 100% - Median Relative Error (%)
      - R² Score (Log-scale and Linear physical domain)
      - Median Relative Error (%) & 95th-Percentile Relative Error (%)
      - Within 5% and Within 10% Tolerance (%)
      - MAE and RMSE in physical Amperes
    """
    results = []
    predictions_log = {}
    predictions_raw = {}
    
    for target in TARGET_COLS:
        model = models[target]
        target_log_name = f"{target}_log"
        
        # 1. Predict in log domain
        pred_log = model.predict(X_eval)
        predictions_log[target] = pred_log
        
        # 2. Reconstruct physical currents in Amperes: I = sign * 10^(log_current)
        sign_actual = y_sign_eval[f"{target}_sign"].values
        pred_raw = sign_actual * (10 ** pred_log)
        predictions_raw[target] = pred_raw
        
        actual_raw = y_raw_eval[target].values
        actual_log = y_log_eval[target_log_name].values
        
        # 3. Calculate R2 scores
        r2_log = r2_score(actual_log, pred_log)
        r2_raw = r2_score(actual_raw, pred_raw)
        
        mae_raw = mean_absolute_error(actual_raw, pred_raw)
        rmse_raw = np.sqrt(mean_squared_error(actual_raw, pred_raw))
        
        # Relative errors using physical noise floor
        floor = 1e-10 if target in ['id', 'is'] else 1e-15
        denom = np.maximum(np.abs(actual_raw), floor)
        rel_errors = (np.abs(pred_raw - actual_raw) / denom) * 100.0
        
        mean_rel_err = np.mean(rel_errors)
        median_rel_err = np.median(rel_errors)
        p95_rel_err = np.percentile(rel_errors, 95)
        
        mean_accuracy = max(0.0, 100.0 - mean_rel_err)
        median_accuracy = max(0.0, 100.0 - median_rel_err)
        within_5pct = (rel_errors <= 5.0).mean() * 100.0
        within_10pct = (rel_errors <= 10.0).mean() * 100.0
        
        results.append({
            "Terminal": target.upper(),
            "Mean Accuracy (%)": f"{mean_accuracy:.2f}%",
            "Median Accuracy (%)": f"{median_accuracy:.2f}%",
            "Median Error (%)": f"{median_rel_err:.2f}%",
            "95th-Pct Error (%)": f"{p95_rel_err:.2f}%",
            "Within 5% Tol (%)": f"{within_5pct:.1f}%",
            "Within 10% Tol (%)": f"{within_10pct:.1f}%",
            "R2 (Log Scale)": round(r2_log, 4),
            "R2 (Linear)": round(r2_raw, 4),
            "MAE (A)": f"{mae_raw:.3e}",
            "RMSE (A)": f"{rmse_raw:.3e}"
        })
        
    df_metrics = pd.DataFrame(results)
    print(f"=======================================================================================================================")
    print(f"                                   {CASE_NAME.upper()} — {split_name.upper()} SET ACCURACY")
    print(f"=======================================================================================================================")
    display(df_metrics)
    
    avg_mean_acc = np.mean([float(r["Mean Accuracy (%)"].replace("%", "")) for r in results])
    avg_med_acc = np.mean([float(r["Median Accuracy (%)"].replace("%", "")) for r in results])
    print(f"[*] Overall {split_name} Mean Accuracy   : {avg_mean_acc:.2f}% (Target: up to 99%)")
    print(f"[*] Overall {split_name} Median Accuracy : {avg_med_acc:.2f}%")
    
    return df_metrics, predictions_raw

# Evaluate on Validation set (Deliverable 4 requirement)
metrics_val, preds_val = evaluate_models(trained_models, X_val, y_log_val, y_raw_val, y_sign_val, "Validation")

# Evaluate on Test set (Deliverable 5 requirement)
metrics_test, preds_test = evaluate_models(trained_models, X_test, y_log_test, y_raw_test, y_sign_test, "Test")


In [ ]:
# 9. Parity Plots: Predicted Current vs. SPICE Reference (Log Axes)
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, target in enumerate(TARGET_COLS):
    ax = axes[idx]
    y_true_abs = np.abs(y_raw_test[target].values)
    y_pred_abs = np.abs(preds_test[target])
    
    # Scatter plot on log scale
    ax.scatter(y_true_abs, y_pred_abs, alpha=0.35, s=12, color='#1f77b4', edgecolors='none', label='Test Samples (1,500)')
    
    # Ideal prediction y = x
    min_val = min(y_true_abs.min(), y_pred_abs.min())
    max_val = max(y_true_abs.max(), y_pred_abs.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1.8, label='Ideal Reference (y = x)')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(f"SPICE Reference |{target.upper()}| (A)", fontsize=11)
    ax.set_ylabel(f"LightGBM Predicted |{target.upper()}| (A)", fontsize=11)
    
    r2_val = metrics_test.loc[metrics_test['Terminal'] == target.upper(), 'R2 (Log Scale)'].values[0]
    med_err = metrics_test.loc[metrics_test['Terminal'] == target.upper(), 'Median Error (%)'].values[0]
    
    ax.set_title(f"Terminal {target.upper()}: R² = {r2_val:.4f} | Med Err = {med_err}", fontsize=12)
    ax.legend(loc='upper left')

plt.suptitle(f"Parity Plots for {CASE_NAME.upper()} (Test Set — 1,500 Samples)", fontsize=14, y=1.01)
plt.tight_layout()

parity_plot_path = os.path.join(PLOTS_DIR, f"{CASE_NAME}_parity_plot.png")
plt.savefig(parity_plot_path, dpi=200, bbox_inches='tight')
print(f"✓ Saved Parity Plot to: {parity_plot_path}")
plt.show()


In [ ]:
# 10. Feature Importance Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, target in enumerate(TARGET_COLS):
    ax = axes[idx]
    model = trained_models[target]
    
    importance = model.booster_.feature_importance(importance_type='gain')
    feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importance}).sort_values('Importance', ascending=True)
    
    # Plot top 12 most important features
    top_feats = feat_df.tail(12)
    ax.barh(top_feats['Feature'], top_feats['Importance'], color='#2ca02c', edgecolor='black', alpha=0.8)
    ax.set_title(f"Top Features (Gain) — {target.upper()}", fontsize=12)
    ax.set_xlabel("Total Gain")

plt.suptitle(f"Physical Feature Importance for {CASE_NAME.upper()} Across Terminals", fontsize=14, y=1.01)
plt.tight_layout()

feat_plot_path = os.path.join(PLOTS_DIR, f"{CASE_NAME}_feature_importance.png")
plt.savefig(feat_plot_path, dpi=200, bbox_inches='tight')
print(f"✓ Saved Feature Importance Plot to: {feat_plot_path}")
plt.show()


In [ ]:
# 11. Model Persistence & Metadata Saving
saved_files = {}
for target, model in trained_models.items():
    model_filename = f"lgbm_{CASE_NAME}_{target}.pkl"
    model_filepath = os.path.join(SAVED_MODELS_DIR, model_filename)
    joblib.dump(model, model_filepath)
    saved_files[target] = model_filepath
    print(f"✓ Saved model: {model_filepath}")

metadata = {
    "case_name": CASE_NAME,
    "feature_columns": FEATURE_COLS,
    "target_columns": TARGET_COLS,
    "epsilon": EPSILON,
    "model_type": "Fine-Tuned LightGBM Regressor (GBDT)",
    "training_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "val_metrics": metrics_val.to_dict(orient="records"),
    "test_metrics": metrics_test.to_dict(orient="records")
}

metadata_path = os.path.join(SAVED_MODELS_DIR, f"{CASE_NAME}_metadata.json")
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"✓ Saved metadata and accuracy metrics: {metadata_path}")

print("\n" + "="*70)
print(f"SUMMARY FOR {CASE_NAME.upper()}:")
print(f"  All 4 terminal models saved in: {SAVED_MODELS_DIR}")
print(f"  Evaluation plots saved in     : {PLOTS_DIR}")
print("="*70)


In [ ]:
# 12. Inference Pipeline Helper (Predict Any PVT Vector)
def predict_leakage_currents(raw_pvt_dict):
    """
    Predicts (Id, Ig, Is, Ib) in Amperes for an input dictionary containing the 12 base PVT parameters.
    Automatically computes derived physical features before passing to the models.
    """
    row = pd.Series(raw_pvt_dict)
    
    # Compute derived physical features
    row['vds'] = row['vd'] - row['vs']
    row['abs_vds'] = abs(row['vds'])
    row['vgs'] = row['vg'] - row['vs']
    row['vgd'] = row['vg'] - row['vd']
    row['vbs'] = row['vb'] - row['vs']
    row['vbd'] = row['vb'] - row['vd']
    row['vov'] = row['vgs'] - row['vth0']
    row['temp_k'] = row['temp'] + 273.15
    row['vt'] = 8.617333e-5 * row['temp_k']
    row['wl_ratio'] = row['w'] / 22e-9
    row['e_ox'] = (row['vg'] - row['vb']) / row['toxe']
    
    input_df = pd.DataFrame([row])[FEATURE_COLS]
    
    predictions = {}
    for target in TARGET_COLS:
        model_path = os.path.join(SAVED_MODELS_DIR, f"lgbm_{CASE_NAME}_{target}.pkl")
        loaded_model = joblib.load(model_path)
        pred_log = loaded_model.predict(input_df)[0]
        
        # Physical sign determination for conduction/tunneling
        if target == 'id':
            sign_val = 1.0 if row['vds'] >= 0 else -1.0
        elif target == 'is':
            sign_val = -1.0 if row['vds'] >= 0 else 1.0
        else:
            sign_val = 1.0
            
        pred_amp = sign_val * (10 ** pred_log)
        predictions[target] = pred_amp
        
    return predictions

# Example test query from test set:
sample_query = {k: X_test.iloc[0][k] for k in BASE_FEATURES}
predicted_currents = predict_leakage_currents(sample_query)

print("Inference query (Base PVT Corner):")
for k, v in sample_query.items():
    print(f"  {k:6s} : {v}")

print("\nPredicted Terminal Currents:")
for t, val in predicted_currents.items():
    print(f"  {t.upper()} : {val:.4e} A")
